# PSF Moments Scatter Plots (v1)

**Author:** Aaron Roodman / Claude  
**Date Created:** 2026-03-18  
**Last Modified:** 2026-03-18  
**Status:** Draft  
**Keywords:** PSF, moments, ellipticity, FWHM, ConsDB  

## Description

Scatter plots of PSF moments vs sequence number for a given night, using the
`PSFMomentsTable` class to query per-CCD data from ConsDB.

Key functionality:
1. Fetch PSF moments (psf_sigma, psf_ixx, psf_iyy, psf_ixy) for 4 CCDs
2. Derive FWHM (arcsec) and ellipticity (e1, e2) per CCD
3. Scatter plots of all quantities vs seq_num

**Output:** Plots  
**Based on:** `common/psf_moments_consdb.py`

<a id='params'></a>
## 1. Parameters

In [ ]:
# ============================================================
# Parameters — All configurable values collected here
# ============================================================

day_obs = 20250415               # Observation date
detectors = [191, 195, 199, 203] # Corner raft CCDs
seq_min = 1
seq_max = 9999

<a id='setup'></a>
## 2. Setup & Imports

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent))
from common.utils import setup_plotting
from common.psf_moments_consdb import PSFMomentsTable

setup_plotting()

# Constants
SIG2FWHM = 2 * np.sqrt(2 * np.log(2))
PIXEL_SCALE = 0.2  # arcsec / pixel

<a id='data'></a>
## 3. Fetch Data

In [ ]:
psf = PSFMomentsTable(
    day_obs=day_obs,
    detectors=detectors,
    seq_min=seq_min,
    seq_max=seq_max,
)
table = psf.fetch()
print(f"{len(table)} visits retrieved for day_obs={day_obs}")
table[:5]

<a id='derived'></a>
## 4. Derive FWHM and Ellipticity

In [ ]:
for det in detectors:
    sigma = table[f"psf_sigma_{det}"]
    ixx = table[f"psf_ixx_{det}"]
    iyy = table[f"psf_iyy_{det}"]
    ixy = table[f"psf_ixy_{det}"]

    # FWHM in arcsec
    table[f"fwhm_{det}"] = sigma * SIG2FWHM * PIXEL_SCALE

    # Ellipticity components
    denom = ixx + iyy
    table[f"e1_{det}"] = (ixx - iyy) / denom
    table[f"e2_{det}"] = 2 * ixy / denom

print("Derived columns added.")

<a id='results'></a>
## 5. Scatter Plots vs Sequence Number

In [ ]:
seq = table["seq_num"]

fig, ax = plt.subplots()
for det in detectors:
    ax.scatter(seq, table[f"fwhm_{det}"], s=10, label=f"det {det}", alpha=0.7)
ax.set_xlabel("seq_num")
ax.set_ylabel("FWHM [arcsec]")
ax.set_title(f"PSF FWHM vs seq_num — day_obs {day_obs}")
ax.legend()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for det in detectors:
    axes[0].scatter(seq, table[f"psf_ixx_{det}"], s=10, label=f"det {det}", alpha=0.7)
    axes[1].scatter(seq, table[f"psf_iyy_{det}"], s=10, label=f"det {det}", alpha=0.7)

axes[0].set_xlabel("seq_num")
axes[0].set_ylabel("psf_ixx [pixels²]")
axes[0].set_title(f"psf_ixx vs seq_num — day_obs {day_obs}")
axes[0].legend()

axes[1].set_xlabel("seq_num")
axes[1].set_ylabel("psf_iyy [pixels²]")
axes[1].set_title(f"psf_iyy vs seq_num — day_obs {day_obs}")
axes[1].legend()

plt.show()

In [ ]:
fig, ax = plt.subplots()
for det in detectors:
    ax.scatter(seq, table[f"psf_ixy_{det}"], s=10, label=f"det {det}", alpha=0.7)
ax.set_xlabel("seq_num")
ax.set_ylabel("psf_ixy [pixels²]")
ax.set_title(f"psf_ixy vs seq_num — day_obs {day_obs}")
ax.legend()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for det in detectors:
    axes[0].scatter(seq, table[f"e1_{det}"], s=10, label=f"det {det}", alpha=0.7)
    axes[1].scatter(seq, table[f"e2_{det}"], s=10, label=f"det {det}", alpha=0.7)

axes[0].set_xlabel("seq_num")
axes[0].set_ylabel("e1")
axes[0].set_title(f"e1 vs seq_num — day_obs {day_obs}")
axes[0].legend()

axes[1].set_xlabel("seq_num")
axes[1].set_ylabel("e2")
axes[1].set_title(f"e2 vs seq_num — day_obs {day_obs}")
axes[1].legend()

plt.show()